In [ ]:
# Load packages 
from functools import reduce
import pandas as pd 
import os

# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [ ]:
# Load ancestry information 
bucket = f"{my_bucket}/data/ancestry_popmad/AoU_KGP_HGDP_MahalMatch_Ancestry.list"
pca = pd.read_csv(bucket, sep ='\t')

In [ ]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
# Save aml case person ids 
bucket = f'{my_bucket}/data/phenotype/aml_person_ids.txt'
aml_person_df.to_csv(bucket, index=False, sep='\t')
# Save ehr person ids 
bucket = f'{my_bucket}/data/phenotype/ehr_person_ids.txt'
ehr_person_df.to_csv(bucket, index=False, sep='\t')

# Create list of quality control information and demographic information 
dfs = [pca.rename(columns={"ID":"person_id"}), ehr_person_df]

# Combine list of quality control information and demographic information 
file = reduce(lambda left, right: pd.merge(left, right, on="person_id", how="inner"), dfs)

# Assign AML phenotype to individuals 
file['aml_phenotype'] = file['person_id'].isin(aml_person_df['person_id']).astype(int)

# Assign AML phenotype to individuals 
file['aml_phenotype'] = file['person_id'].isin(aml_person_df['person_id']).astype(int)

# Calculate phenotype counts per ancestry 
bucket = f'{my_bucket}/data/phenotype/phenotype_counts.txt'
file[['aml_phenotype','superpopSD3']].value_counts().reset_index().to_csv(bucket, index=False, sep='\t')
